In [62]:
import os
import pandas as pd
import requests
import time
import re
import numpy as np
from io import StringIO
from Bio import SeqIO 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from sentence_transformers import SentenceTransformer
from sklearn.metrics import silhouette_score


**Fetch Descriptions**

In [63]:
fasta_path = '../data/raw/beta1.fasta'
raw_ids = []

for record in SeqIO.parse(fasta_path, "fasta"):
    header = record.description
    match = re.search(r'UniRef\d+_([A-Z0-9]+)', header)
    if match:
        extracted_id = match.group(1)
        if not extracted_id.startswith('UPI'):
            raw_ids.append(extracted_id)

raw_ids = list(set(raw_ids))

cache_file = '../data/processed/uniprot_all_fields_cache.csv'
os.makedirs(os.path.dirname(cache_file), exist_ok=True)

if os.path.exists(cache_file):
    print(f"Found existing cache! Loading from {cache_file}...")
    df_raw = pd.read_csv(cache_file)
    df_raw = df_raw.fillna('')
else:
    print("Fetching expanded data from UniProt API...")
    def chunk_list(lst, n):
        for i in range(0, len(lst), n):
            yield lst[i:i + n]
            
    batches = list(chunk_list(raw_ids, 200))
    all_dataframes = []
    
    for i, batch in enumerate(batches):
        print(f"Fetching batch {i+1}/{len(batches)}...")
        joined_ids = ",".join(batch)
        
        url = f"https://rest.uniprot.org/uniprotkb/accessions?accessions={joined_ids}&format=tsv&fields=accession,cc_function,keyword,go,gene_names,organism_name,protein_name,ft_binding,cc_pathway,cc_activity_regulation"
        try:
            response = requests.get(url)
            if response.status_code == 200:
                df_batch = pd.read_csv(StringIO(response.text), sep='\t')
                all_dataframes.append(df_batch)
            else:
                print(f"API Error on batch {i+1}: Status {response.status_code}")
        except Exception as e:
            print(f"Error on batch {i+1}: {e}")
            
        time.sleep(1) 
        
    if all_dataframes:
        df_raw = pd.concat(all_dataframes, ignore_index=True)
        df_raw = df_raw.fillna('')
        df_raw.to_csv(cache_file, index=False)
        print(f"Saved expanded fields to {cache_file}")
    else:
        df_raw = pd.DataFrame()

if not df_raw.empty:
    print("\n--- DATA AVAILABILITY REPORT ---")
    print(f"Total Proteins Retrieved: {len(df_raw)}")
    
    def count_valid(col_name):

        if col_name in df_raw.columns:
            return sum(df_raw[col_name].astype(str).str.strip() != '')
        return 0

    print(f"Function [CC]:       {count_valid('Function [CC]')}")
    print(f"Binding Site [FT]:   {count_valid('Binding site')}")
    print(f"Pathway [CC]:        {count_valid('Pathway')}")
    print(f"Activity Reg [CC]:   {count_valid('Activity regulation')}")
    print(f"Keywords:            {count_valid('Keywords')}")
    print(f"Gene Ontology:       {count_valid('Gene Ontology (GO)')}")
    print(f"Gene Names:          {count_valid('Gene Names')}")
    print(f"Organism:            {count_valid('Organism')}")
    print(f"Protein Names:       {count_valid('Protein names')}")

Found existing cache! Loading from ../data/processed/uniprot_all_fields_cache.csv...

--- DATA AVAILABILITY REPORT ---
Total Proteins Retrieved: 1037
Function [CC]:       167
Binding Site [FT]:   2
Pathway [CC]:        0
Activity Reg [CC]:   0
Keywords:            1037
Gene Ontology:       1037
Gene Names:          767
Organism:            1037
Protein Names:       1037


**Sentence BERT for embbeding functions**

In [64]:
cache_file = '../data/processed/uniprot_all_fields_cache.csv'
df = pd.read_csv(cache_file).fillna('')

print("Processing Semantic Text with Sentence-BERT...")
functions = df['Function [CC]'].astype(str).tolist()

sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
sbert_embeddings = sbert_model.encode(functions, show_progress_bar=True)

sbert_sim = cosine_similarity(sbert_embeddings)
sbert_dist = np.clip(1 - sbert_sim, 0, None)


Processing Semantic Text with Sentence-BERT...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 250.00it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 33/33 [00:01<00:00, 28.52it/s]


In [65]:
print(f"Shape of SBERT embeddings matrix: {sbert_embeddings.shape}")

print(" --- SBERT EMBEDDING SAMPLES ---")
count = 0

for i, (text, vector) in enumerate(zip(functions, sbert_embeddings)):
    if text.strip() != "":
        print(f"Protein Index: {i}")
        print(f"Text: {text[:150]}...")
        print(f"Vector (First 5 dims): {vector[:5]}")
        print("-" * 60)
        count += 1
        
    if count >= 3:
        break

Shape of SBERT embeddings matrix: (1037, 384)
 --- SBERT EMBEDDING SAMPLES ---
Protein Index: 6
Text: FUNCTION: Beta-adrenergic receptors mediate the catecholamine-induced activation of adenylate cyclase through the action of G proteins. This receptor ...
Vector (First 5 dims): [-0.03462449 -0.00441799 -0.04139499  0.07411824  0.03278499]
------------------------------------------------------------
Protein Index: 10
Text: FUNCTION: Beta-adrenergic receptors mediate the catecholamine-induced activation of adenylate cyclase through the action of G proteins. Beta-3 is invo...
Vector (First 5 dims): [-0.0773917  -0.07787169 -0.06384401  0.02881473 -0.05608949]
------------------------------------------------------------
Protein Index: 28
Text: FUNCTION: Receptor for gastrin and cholecystokinin. The CCK-B receptors occur throughout the central nervous system where they modulate anxiety, analg...
Vector (First 5 dims): [-0.02633777 -0.05266939 -0.09493968  0.11363614 -0.00792759]
-----------

**Use tf-idf for other features**

In [66]:
print("Processing Biological Tags with TF-IDF...")
df['combined_tags'] = (
    df['Keywords'] + " " + 
    df['Gene Ontology (GO)'] + " " + 
    df['Gene Names'] + " " + 
    df['Protein names']
).str.replace(';', ' ')

tfidf_vec = TfidfVectorizer(max_df=0.95, min_df=2, stop_words='english')
tfidf_embeddings = tfidf_vec.fit_transform(df['combined_tags'])

tfidf_sim = cosine_similarity(tfidf_embeddings)
tfidf_dist = np.clip(1 - tfidf_sim, 0, None)

Processing Biological Tags with TF-IDF...


In [67]:
print(f"Shape of TF-IDF matrix: {tfidf_embeddings.shape}")

feature_names = tfidf_vec.get_feature_names_out()

print("--- TF-IDF EMBEDDING SAMPLES ---")
for i in range(3):
    text = df['combined_tags'].iloc[i]
    print(f"Protein Index: {i}")
    
    print(f"Raw Tags: {text[:150]}...")
    
    row_vector = tfidf_embeddings[i].toarray()[0]
    
    non_zero_indices = row_vector.nonzero()[0]
    
    top_indices = non_zero_indices[np.argsort(row_vector[non_zero_indices])[::-1]][:5]
    top_words = {feature_names[idx]: round(row_vector[idx], 4) for idx in top_indices}
    
    print(f"Top 5 mathematically weighted words: {top_words}")
    print("-" * 80)


Shape of TF-IDF matrix: (1037, 514)
--- TF-IDF EMBEDDING SAMPLES ---
Protein Index: 0
Raw Tags: Cell membrane Disulfide bond G-protein coupled receptor Membrane Receptor Reference proteome Transducer Transmembrane Transmembrane helix plasma membr...
Top 5 mathematically weighted words: {'adrb3a': np.float64(0.42), 'epinephrine': np.float64(0.2986), 'adrenergic': np.float64(0.2743), 'like': np.float64(0.2583), '0015052': np.float64(0.248)}
--------------------------------------------------------------------------------
Protein Index: 1
Raw Tags: Cell membrane G-protein coupled receptor Membrane Receptor Reference proteome Transducer Transmembrane Transmembrane helix plasma membrane [GO:0005886...
Top 5 mathematically weighted words: {'receptors': np.float64(0.3462), 'profile': np.float64(0.3462), 'family': np.float64(0.3434), 'domain': np.float64(0.3323), '0004930': np.float64(0.3317)}
--------------------------------------------------------------------------------
Protein Index: 2
Raw 

**Clustering based on previous parts**

In [ ]:
print("Merging into Hybrid Ensemble Matrix...")
hybrid_dist = (0.7 * tfidf_dist) + (0.3 * sbert_dist)
hybrid_dist = np.clip(hybrid_dist, 0, None)
np.fill_diagonal(hybrid_dist, 0)

print("Finding optimal number of clusters (Testing k=10 to k=60)...")

best_k = 10
best_score = -1
k_range = range(10, 61, 2)
scores = []

for k in k_range:
    test_model = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average')
    test_labels = test_model.fit_predict(hybrid_dist)
    
    score = silhouette_score(hybrid_dist, test_labels, metric='precomputed')
    scores.append(score)


    
    if score > best_score:
        best_score = score
        best_k = k
        
print(f"The optimal number of clusters is: k = {best_k} (Silhouette Score: {best_score:.4f})\n")

print("Applying optimal clustering...")
final_model = AgglomerativeClustering(n_clusters=best_k, metric='precomputed', linkage='average')
df['hybrid_cluster'] = final_model.fit_predict(hybrid_dist)

print("\nFinal Cluster Sizes:")
print(df['hybrid_cluster'].value_counts().sort_index().head(10)) 
print("...\n")

print("Some ranom clurstrs:")
print("-" * 80)

valid_clusters = [c for c in df['hybrid_cluster'].unique() if (df['hybrid_cluster'] == c).sum() >= 5]
sample_clusters = np.random.choice(valid_clusters, 3, replace=False)

for cluster_id in sample_clusters:
    print(f"\nCLUSTER {cluster_id}")
    cluster_data = df[df['hybrid_cluster'] == cluster_id]
    
    for idx, row in cluster_data.head(4).iterrows():
        protein_name = row.get('Protein names', 'Unknown Protein')
        gene_name = row.get('Gene Names', 'Unknown Gene')
        print(f"  🔹 {protein_name[:80]}... | Gene: {gene_name}")
    
    print(f"  (Total proteins in this cluster: {len(cluster_data)})")
    print("-" * 80)

Merging into Hybrid Ensemble Matrix...
Finding optimal number of clusters (Testing k=10 to k=60)...
The optimal number of clusters is: k = 60 (Silhouette Score: 0.5062)

Applying optimal clustering...

Final Cluster Sizes:
hybrid_cluster
0     2
1    20
2     4
3    44
4    15
5    15
6    18
7    38
8    60
9    24
Name: count, dtype: int64
...

Some ranom clurstrs:
--------------------------------------------------------------------------------

CLUSTER 11
  🔹 Adrenoceptor beta 3... | Gene: ADRB3
  🔹 Beta-2 adrenergic receptor (Beta-2 adrenoreceptor)... | Gene: ADRB2
  🔹 Beta-2 adrenergic receptor (Beta-2 adrenoreceptor)... | Gene: ADRB2
  🔹 Adrenoceptor beta 3... | Gene: ADRB3
  (Total proteins in this cluster: 8)
--------------------------------------------------------------------------------

CLUSTER 8
  🔹 DRD5L protein... | Gene: Dl UROIND_R10810
  🔹 DRD5L protein... | Gene: Dl CHLAEN_R00056
  🔹 Beta-2 adrenergic receptor (Beta-2 adrenoreceptor)... | Gene: 
  🔹 Octopamine recepto